<center><h1>Prado_Mario_HW3</h1></center>
<br>
<br>

Name: Mario Prado
<br>
Github Username: mrpradooo
<br>
USC ID: 4223775338

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [20]:
import shutil
import os
import glob
import re
import pandas as pd
from scipy.stats import bootstrap
import numpy as np

Get the AReM Data Set

Downloaded through Mac finder

### (b) Test and Train Data

In [21]:
def numeric_sort_key(filename):

    match = re.search(r'\d+', filename)
    return int(match.group()) if match else 0

activity_folders = ['bending1', 'bending2', 'cycling', 'lying', 'sitting', 'standing', 'walking']
base_path = '../data/AReM/'


for activity in activity_folders:
    folder_path = os.path.join(base_path, activity)
    files = sorted(os.listdir(folder_path))


    if activity in ['bending1', 'bending2']:
        test_files = files[:2]
    else:
        test_files = files[:3]


    train_files = [f for f in files if f not in test_files]

    for f in test_files:
        new_name = f"{activity}_{f}"
        shutil.copy(os.path.join(folder_path, f), os.path.join('../data/test', new_name))
    for f in train_files:
        new_name = f"{activity}_{f}"
        shutil.copy(os.path.join(folder_path, f), os.path.join('../data/train', new_name))


print("Train and Test Ready")



Train and Test Ready


### (c) Feature Extraction

#### i. Research

Minimum, maximum, mean, standard deviation, variane, range, skewness

#### ii. Extraction

In [22]:
def read_sensor_file(file_path):
    rows = []
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = [p for p in re.split(r'[,\s]+', line.strip()) if p]
            if len(parts) >= 7:
                rows.append([float(x) for x in parts[:7]])
    return pd.DataFrame(rows, columns=['time', 'avg_rss12', 'var_rss12',
                                       'avg_rss13', 'var_rss13',
                                       'avg_rss23', 'var_rss23'])

In [23]:
def feature_extractor(file_path, activity_label):
    df = read_sensor_file(file_path)

    time_series_cols = ['avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']
    
    features = {}
    for i, col in enumerate(time_series_cols, start=1):
        features[f'min{i}'] = df[col].min()
        features[f'max{i}'] = df[col].max()
        features[f'mean{i}'] = df[col].mean()
        features[f'median{i}'] = df[col].median()
        features[f'std{i}'] = df[col].std()
        features[f'1st_quart{i}'] = df[col].quantile(0.25)
        features[f'3rd_quart{i}'] = df[col].quantile(0.75)
    
    features['activity'] = activity_label
    return features

In [24]:
def get_activity(filename):
    match = re.match(r'([a-zA-Z0-9]+)_dataset', filename)
    return match.group(1) if match else 'unknown'

In [25]:
train_features = []

for filename in os.listdir('../data/train'):
    if not filename.endswith('.csv'):
        continue
    file_path = os.path.join('../data/train', filename)
    activity = get_activity(filename)
    try:
        features = feature_extractor(file_path, activity)
        train_features.append(features)
    except Exception as e:
        print(f"FAILED on {filename}: {e}")

train_feature_df = pd.DataFrame(train_features)
print(train_feature_df.shape)

(69, 43)


In [26]:
test_features = []

for filename in os.listdir('../data/test'):
    if not filename.endswith('.csv'):
        continue
    file_path = os.path.join('../data/test', filename)
    activity = get_activity(filename)
    try:
        features = feature_extractor(file_path, activity)
        test_features.append(features)
    except Exception as e:
        print(f"FAILED on {filename}: {e}")

test_feature_df = pd.DataFrame(test_features)
print(test_feature_df.shape)

(19, 43)


#### iii. Standard Deviation

In [27]:
cols_feature = [col for col in train_feature_df.columns if col != 'activity']

results_bootstrap = {}

for col in cols_feature:
    data = (train_feature_df[col].values,)  
    
    res = bootstrap(data, np.std, confidence_level=0.90, n_resamples=1000, method='percentile')
    
    results_bootstrap[col] = {
        'std_estimate': np.std(train_feature_df[col].values),
        'ci_low': res.confidence_interval.low,
        'ci_high': res.confidence_interval.high
    }

bootstrap_df = pd.DataFrame(results_bootstrap).T  
bootstrap_df

,std_estimate,ci_low,ci_high
min1,9.184253,7.916414,10.354972
max1,3.951251,2.918636,4.866270
mean1,4.807935,4.306051,5.295520
median1,4.933215,4.377654,5.456210
std1,1.863093,1.625116,2.063047
1st_quart1,5.818462,5.233929,6.318410
3rd_quart1,4.597145,3.745200,5.257459
min2,0.000000,0.000000,0.000000
max2,5.120343,4.621058,5.465828
mean2,1.579090,1.383679,1.717767


In [ ]:
print(train_feature_df.isna().sum().sum())  
print(test_feature_df.isna().sum().sum())

0
0


#### iv. Select Features

I decided to go with min, mean, max becuase I feel as this is a good combo to get the full picture of my data, It will show me the extremes on both sides while also giving me the average value.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regresssion has the smaller RSS and this is due to the fact that the linear model is a special case of the cubic model, so the linear RSS can never be smaller than the cubic RSS, furthermore the cubic RSS can be even smaller because the cubic model has two extra coefficeints that can take into account more noise.

### (b) Linear Test

If we're using test instead, than the linear model has a smaller RSS as the cubic model will use its extra coefficients to fit to noise that doesn't carry over and thus creating a bigger RSS.

### (c) Not Linear Train

Even though the relationship is not linear, because the linear model is still a special case of the cubic model the cubic model RSS can never be bigger than that of the lienar model's but if the cubic model catches on to some noise then the cubic RSS will be smaller.

### (d) Not Linear Testing

I don't think we can tell with the information given, we need to know how linear the truth is, if it's almost linear than the linear model will do better but if it has a big enough curve then the cubic model will do better.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 